# MLP com Softmax — AlignFit TCC

> **Disciplina:** Redes Neurais / Aprendizado de Máquina  
> **Projeto:** AlignFit — Sistema de Análise Biomecânica de Exercícios  
> **Modelo:** Multi-Layer Perceptron (MLP) com ativação Softmax na saída  
> **Tarefa:** Classificação multiclasse — identificação do exercício executado

---

## 1. Contexto e Motivação

O AlignFit é um sistema baseado em IA e Visão Computacional que analisa vídeos de exercícios de musculação. O pipeline extrai keypoints (pontos-chave) do corpo humano usando YOLOv8 Pose e alimenta modelos de ML para:

1. **Identificar qual exercício** está sendo executado
2. **Classificar a execução** como correta ou incorreta

O modelo atual usa XGBoost. Neste notebook, implementamos um **MLP Classificador** como modelo alternativo — com foco em entender e aplicar a função de ativação **Softmax**, que é a escolha natural para classificação multiclasse em redes neurais.

**Os 4 exercícios suportados:**
- Desenvolvimento de Ombro
- Elevação Lateral
- Rosca Direta (Bíceps)
- Agachamento

---

## 2. Fundamentos Teóricos

### 2.1 Perceptron → MLP

O notebook `MY_FIRST_PERCEPTRON.ipynb` mostrou a base: um neurônio que calcula uma **soma ponderada** das entradas mais um bias e aplica uma **função de ativação** para gerar uma saída:

```
z = w₁·x₁ + w₂·x₂ + ... + wₙ·xₙ + b
saída = f(z)
```

O **Perceptron** usa uma função de passo (step function) — só funciona para classificação binária e não é diferenciável, o que impede o uso de gradiente descendente.

O **MLP (Multi-Layer Perceptron)** empilha múltiplas camadas de neurônios com funções de ativação diferenciáveis (como ReLU nas camadas intermediárias), permitindo aprender padrões complexos e não-lineares. O treinamento usa **backpropagation** com gradiente descendente.

### 2.2 Por que Softmax?

Para classificação **binária** (correto/errado), a camada de saída usa um único neurônio com **Sigmoid**, que produz uma probabilidade entre 0 e 1.

Para classificação **multiclasse** (4 exercícios), precisamos de uma distribuição de probabilidade sobre todas as classes. É aqui que entra o **Softmax**.

### 2.3 Fórmula do Softmax

Dado um vetor de scores brutos `z = [z₁, z₂, ..., zₖ]` — um score por classe — o Softmax converte esses scores em probabilidades:

```
          e^(zᵢ)
P(i) = -----------
        Σ e^(zⱼ)
          j=1..k
```

**Propriedades importantes:**
- `0 < P(i) < 1` para todo i
- `Σ P(i) = 1` (soma das probabilidades = 100%)
- A classe com maior score recebe a maior probabilidade
- O exponencial amplifica diferenças: scores maiores dominam muito mais

**Por que exponencial e não normalização simples?**  
A exponencial garante que todos os valores são positivos (mesmo scores negativos viram positivos) e é matematicamente compatível com a função de perda **Cross-Entropy**, formando um par que tem derivada muito limpa para o backpropagation.

### 2.4 Função de Perda: Cross-Entropy

Com Softmax na saída, a função de perda padrão é a **Cross-Entropy Categórica**:

```
Loss = -Σ yᵢ · log(P(i))
```

Onde `yᵢ` é 1 para a classe correta e 0 para as demais (one-hot encoding).  
Isso penaliza fortemente quando o modelo dá baixa probabilidade à classe correta.

### 2.5 Arquitetura do MLP neste notebook

```
Input Layer    → 136 features (média+std+min+max de 34 keypoints x e y)
Hidden Layer 1 → 128 neurônios, ativação ReLU
Hidden Layer 2 → 64 neurônios,  ativação ReLU
Hidden Layer 3 → 32 neurônios,  ativação ReLU
Output Layer   → 4 neurônios,   ativação Softmax → [P(desenvolvimento), P(elevacao), P(biceps), P(agachamento)]
```

**ReLU nas camadas intermediárias:** `f(z) = max(0, z)` — resolve o problema de vanishing gradient e é computacionalmente simples.  
**Softmax na camada de saída:** transforma os 4 scores em probabilidades que somam 1.

---

## 3. Demonstração Manual do Softmax

Antes de usar o sklearn, vamos implementar o Softmax do zero — assim como a professora fez com o Perceptron manual.

In [ ]:
import numpy as np

# =========================================================
# SOFTMAX MANUAL — implementação do zero
# =========================================================

def softmax(z):
    """
    Calcula a função Softmax sobre um vetor de scores.
    
    O truque numérico: subtraímos o máximo antes de exponenciar.
    Isso evita overflow (e^10000 → inf) sem mudar o resultado,
    porque dividir numerador e denominador por e^max cancela.
    
    Prova: e^(z-max) / Σ e^(zⱼ-max)  =  e^z / Σ e^(zⱼ)  (idêntico)
    """
    z_stable = z - np.max(z)          # estabilidade numérica
    exp_z    = np.exp(z_stable)        # e^zᵢ para cada i
    return exp_z / np.sum(exp_z)       # normaliza: divide pela soma total


# -----------------------------------------------------------
# Exemplo com 4 classes (os 4 exercícios do AlignFit)
# -----------------------------------------------------------
# Imagine que a rede neural gerou estes scores brutos (logits)
# para um vídeo de desenvolvimento de ombro:

classes = ["desenvolvimento", "elevacao_lateral", "biceps", "agachamento"]

scores_brutos = np.array([3.2, 1.1, 0.5, -0.8])   # logits da rede

probabilidades = softmax(scores_brutos)

print("=" * 50)
print("DEMONSTRAÇÃO DO SOFTMAX")
print("=" * 50)

print("\nScores brutos (logits):")
for cls, score in zip(classes, scores_brutos):
    print(f"  {cls:20s}: {score:6.2f}")

print("\nApós Softmax (probabilidades):")
for cls, prob in zip(classes, probabilidades):
    barra = "█" * int(prob * 40)
    print(f"  {cls:20s}: {prob:.4f}  ({prob*100:5.1f}%)  {barra}")

print(f"\nSoma das probabilidades: {probabilidades.sum():.6f}  (deve ser 1.0)")
print(f"Classe prevista: {classes[np.argmax(probabilidades)]}")

# -----------------------------------------------------------
# Mostrando o efeito do exponencial
# -----------------------------------------------------------
print("\n" + "=" * 50)
print("POR QUE EXPONENCIAL? — comparação com normalização simples")
print("=" * 50)

scores_positivos = scores_brutos - scores_brutos.min()   # todos >= 0
normalizacao_simples = scores_positivos / scores_positivos.sum()

print("\nNormalização linear (simples):")
for cls, p in zip(classes, normalizacao_simples):
    print(f"  {cls:20s}: {p:.4f}  ({p*100:5.1f}%)")

print("\nSoftmax (com exponencial):")
for cls, p in zip(classes, probabilidades):
    print(f"  {cls:20s}: {p:.4f}  ({p*100:5.1f}%)")

print("\n→ O Softmax é muito mais 'decidido': amplifica a diferença entre scores.")
print("  Isso ajuda o modelo a ter alta confiança quando o padrão é claro.")

In [ ]:
# =========================================================
# CROSS-ENTROPY MANUAL
# =========================================================

def cross_entropy(y_true_onehot, y_prob):
    """
    Calcula a Cross-Entropy entre o rótulo verdadeiro (one-hot)
    e as probabilidades previstas pelo Softmax.
    
    Loss = -Σ yᵢ · log(P(i))
    Como yᵢ = 0 para todas as classes exceto a correta,
    isso simplifica para: Loss = -log(P(classe_correta))
    """
    eps = 1e-12   # evita log(0)
    return -np.sum(y_true_onehot * np.log(y_prob + eps))


print("=" * 50)
print("DEMONSTRAÇÃO DA CROSS-ENTROPY")
print("=" * 50)

# Rótulo verdadeiro: é desenvolvimento (índice 0)
y_true = np.array([1, 0, 0, 0])    # one-hot para "desenvolvimento"

# Cenário 1: modelo CONFIANTE e CERTO
prob_certo_confiante   = np.array([0.92, 0.04, 0.03, 0.01])

# Cenário 2: modelo INCERTO
prob_incerto           = np.array([0.40, 0.30, 0.20, 0.10])

# Cenário 3: modelo ERRADO
prob_errado            = np.array([0.05, 0.85, 0.07, 0.03])

cenarios = [
    ("Confiante e correto",  prob_certo_confiante),
    ("Incerto",              prob_incerto),
    ("Errado",               prob_errado),
]

for nome, prob in cenarios:
    loss = cross_entropy(y_true, prob)
    print(f"\n  {nome}")
    print(f"  P(desenvolvimento) = {prob[0]:.2f}")
    print(f"  Loss = -log({prob[0]:.2f}) = {loss:.4f}")

print("\n→ Quanto maior o erro, maior a loss — o modelo aprende a minimizá-la.")

---

## 4. Pipeline de Dados

Mesma lógica de feature engineering usada no XGBoost — garante comparação justa entre os modelos.

In [ ]:
import pandas as pd
import numpy as np

# =========================================================
# CARREGAMENTO DO DATASET
# =========================================================

DATASET_PATH = r"..\datasets\raw-dataset.csv"

print("Carregando dataset...")
df = pd.read_csv(DATASET_PATH, sep=",")

print(f"Shape: {df.shape}")
print(f"Exercícios: {df['tipo_exercicio'].unique()}")
print(f"Execuções:  {df['tipo_execucao'].unique()}")

# Quantos vídeos únicos por exercício?
print("\nVídeos por exercício:")
print(df.groupby('tipo_exercicio')['video'].nunique())

In [ ]:
from sklearn.preprocessing import LabelEncoder

# =========================================================
# FEATURE ENGINEERING
# Mesma lógica do BuildExerciseClassifierModel.py
# Agrupamos por vídeo e extraímos: média, std, min, max
# de cada coordenada x e y dos keypoints
# =========================================================

# Colunas de features: todas que terminam em _x ou _y
feature_columns = [
    col for col in df.columns
    if col.endswith("_x") or col.endswith("_y")
]

print(f"Features de keypoint: {len(feature_columns)} colunas")
print(f"Features totais após média+std+min+max: {len(feature_columns) * 4}")

# Encode dos rótulos
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["tipo_exercicio"])

print(f"\nClasses codificadas:")
for idx, cls in enumerate(label_encoder.classes_):
    print(f"  {idx} → {cls}")

# =========================================================
# EXTRAÇÃO DE FEATURES POR VÍDEO
# =========================================================

X = []
y = []

for video_name, group in df.groupby("video"):
    features = []
    features.extend(group[feature_columns].mean().tolist())           # média
    features.extend(group[feature_columns].std().fillna(0).tolist())  # desvio padrão
    features.extend(group[feature_columns].min().tolist())            # mínimo
    features.extend(group[feature_columns].max().tolist())            # máximo
    X.append(features)
    y.append(group["label"].iloc[0])

X = np.array(X)
y = np.array(y)

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nDistribuição de classes:")
for idx, cls in enumerate(label_encoder.classes_):
    count = (y == idx).sum()
    print(f"  {cls}: {count} vídeos")

---

## 5. Normalização dos Dados

**O XGBoost não precisa de normalização** — ele usa árvores de decisão, que são invariantes à escala.

**O MLP precisa de normalização** — os pesos são inicializados em escala pequena e o gradiente descendente converge muito mais rápido quando as features estão na mesma escala. Sem isso, features com valores grandes (ex: coordenadas em pixels de 0 a 1000) dominam o aprendizado.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# =========================================================
# SPLIT TREINO / TESTE
# stratify=y garante proporção igual de classes nos dois conjuntos
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Treino: {X_train.shape}")
print(f"Teste:  {X_test.shape}")

# =========================================================
# NORMALIZAÇÃO — StandardScaler (Z-score)
# Fórmula: x_normalizado = (x - média) / desvio_padrão
# Resultado: média 0, desvio padrão 1
#
# ATENÇÃO: fit() apenas no treino, transform() em ambos.
# Se fizéssemos fit() no teste também, haveria data leakage
# (o modelo veria informação do futuro durante o treino).
# =========================================================

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)    # aprende média/std do treino e transforma
X_test_scaled  = scaler.transform(X_test)          # aplica a mesma transformação no teste

print(f"\nApós normalização:")
print(f"  Média das features (treino): {X_train_scaled.mean():.4f}  (esperado ≈ 0)")
print(f"  Std  das features (treino): {X_train_scaled.std():.4f}   (esperado ≈ 1)")

---

## 6. Treinamento do MLP com Softmax

O `MLPClassifier` do scikit-learn usa automaticamente **Softmax + Cross-Entropy** quando há mais de 2 classes — que é exatamente o nosso caso (4 exercícios).

Internamente, o sklearn chama isso de `loss='log_loss'` (Cross-Entropy) com saída multiclasse.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import time

# =========================================================
# DEFINIÇÃO DO MLP
# =========================================================
# Arquitetura:
#   hidden_layer_sizes=(128, 64, 32) → 3 camadas ocultas
#   Camada de entrada:  tamanho automático (= n° de features)
#   Camada de saída:    tamanho automático (= n° de classes)
#                       com ativação Softmax implícita
#
# activation='relu'  → ReLU nas camadas ocultas
# solver='adam'      → Gradiente descendente adaptativo (mais rápido que SGD puro)
# max_iter=500       → Número máximo de épocas
# early_stopping=True→ Para o treino quando a val_loss para de melhorar
#                      (evita overfitting)
# validation_fraction→ 15% do treino usado como validação interna
# =========================================================

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    alpha=0.001,              # L2 regularization (penaliza pesos grandes → reduz overfitting)
    learning_rate_init=0.001, # taxa de aprendizado inicial do Adam
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,      # para após 20 épocas sem melhora na validação
    random_state=42,
    verbose=True
)

print("Arquitetura do MLP:")
print(f"  Entrada:        {X_train_scaled.shape[1]} features")
print(f"  Camada oculta 1: 128 neurônios (ReLU)")
print(f"  Camada oculta 2: 64 neurônios  (ReLU)")
print(f"  Camada oculta 3: 32 neurônios  (ReLU)")
print(f"  Saída:          {len(label_encoder.classes_)} neurônios (Softmax)")
print(f"  Classes: {list(label_encoder.classes_)}")

print("\nTreinando...\n")
inicio = time.time()

mlp.fit(X_train_scaled, y_train)

tempo = time.time() - inicio
print(f"\nTreinamento concluído em {tempo:.1f}s")
print(f"Épocas executadas: {mlp.n_iter_}")

---

## 7. Avaliação do Modelo

In [ ]:
# =========================================================
# PREDIÇÃO E MÉTRICAS
# =========================================================

y_pred  = mlp.predict(X_test_scaled)
y_proba = mlp.predict_proba(X_test_scaled)   # probabilidades Softmax

acc = accuracy_score(y_test, y_pred)

print("=" * 55)
print(f"ACURÁCIA NO TESTE: {acc:.4f} ({acc*100:.1f}%)")
print("=" * 55)

print("\nRELATÓRIO DE CLASSIFICAÇÃO:")
print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_
))

# Glossário das métricas:
print("Glossário:")
print("  Precision: de todos que o modelo disse 'X', quantos eram realmente 'X'?")
print("  Recall:    de todos que eram realmente 'X', quantos o modelo identificou?")
print("  F1-score:  média harmônica de precision e recall")
print("  Support:   número de amostras reais de cada classe no conjunto de teste")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

# =========================================================
# MATRIZ DE CONFUSÃO
# =========================================================

cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 5))

im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)

classes = label_encoder.classes_
tick_marks = np.arange(len(classes))

ax.set_xticks(tick_marks)
ax.set_yticks(tick_marks)
ax.set_xticklabels(classes, rotation=30, ha="right")
ax.set_yticklabels(classes)

thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j, i, str(cm[i, j]),
            ha="center", va="center",
            color="white" if cm[i, j] > thresh else "black",
            fontsize=13
        )

ax.set_ylabel("Rótulo Real")
ax.set_xlabel("Rótulo Previsto")
ax.set_title("Matriz de Confusão — MLP Softmax")
plt.tight_layout()
plt.show()

print("\nInterpretação:")
print("  Diagonal principal (azul escuro) = acertos")
print("  Fora da diagonal = confusões entre exercícios")

In [ ]:
# =========================================================
# CURVA DE PERDA (Loss Curve) — acompanha o treinamento
# =========================================================

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(mlp.loss_curve_,      label="Loss de treino",   color="steelblue", linewidth=2)
ax.plot(mlp.validation_scores_, label="Score de validação", color="coral",     linewidth=2, linestyle="--")

ax.set_xlabel("Época")
ax.set_ylabel("Loss / Score")
ax.set_title("Curva de Aprendizado — MLP com Softmax")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nO que observar:")
print("  - Loss de treino deve cair monotonicamente")
print("  - Se a loss de validação começa a subir enquanto a de treino cai → overfitting")
print("  - O early_stopping interrompeu o treino para evitar isso")

In [ ]:
# =========================================================
# VISUALIZAÇÃO DAS SAÍDAS SOFTMAX
# Mostra as probabilidades para algumas amostras do teste
# =========================================================

n_exemplos = min(6, len(X_test_scaled))

fig, axes = plt.subplots(1, n_exemplos, figsize=(3 * n_exemplos, 4))
if n_exemplos == 1:
    axes = [axes]

cores = ["#4CAF50" if y_pred[i] == y_test[i] else "#F44336" for i in range(n_exemplos)]

for i, ax in enumerate(axes):
    probs  = y_proba[i]
    correto = label_encoder.classes_[y_test[i]]
    previsto = label_encoder.classes_[y_pred[i]]

    barras = ax.barh(label_encoder.classes_, probs, color="steelblue", edgecolor="white")
    ax.barh(
        label_encoder.classes_[y_test[i]],
        probs[y_test[i]],
        color="gold",
        edgecolor="white"
    )
    ax.set_xlim(0, 1)
    ax.set_xlabel("Probabilidade")
    ax.set_title(
        f"Amostra {i+1}\n"
        f"Real: {correto}\n"
        f"Previsto: {previsto}",
        color=cores[i], fontsize=8
    )
    ax.axvline(0.5, color="gray", linestyle="--", linewidth=0.8)

plt.suptitle("Probabilidades Softmax — Verde=Acerto / Vermelho=Erro / Dourado=Classe real", y=1.02)
plt.tight_layout()
plt.show()

---

## 8. Comparação: MLP Softmax vs. XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# =========================================================
# TREINAR XGBOOST (mesmos dados, sem normalização)
# =========================================================

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss",
    use_label_encoder=False
)

xgb.fit(X_train, y_train)    # XGBoost usa X sem normalização
y_pred_xgb = xgb.predict(X_test)
acc_xgb = accuracy_score(y_test, y_pred_xgb)

# =========================================================
# COMPARAÇÃO
# =========================================================

acc_mlp = accuracy_score(y_test, y_pred)

print("=" * 55)
print("COMPARATIVO DE MODELOS — Classificador de Exercício")
print("=" * 55)
print(f"  MLP Softmax (sklearn):  {acc_mlp:.4f}  ({acc_mlp*100:.1f}%)")
print(f"  XGBoost:                {acc_xgb:.4f}  ({acc_xgb*100:.1f}%)")
print("=" * 55)

vencedor = "MLP Softmax" if acc_mlp > acc_xgb else "XGBoost"
print(f"\nMelhor acurácia no teste: {vencedor}")

print("\nObservações:")
print("  - XGBoost tende a ser superior com datasets pequenos (poucas amostras)")
print("  - MLP melhora proporcionalmente com mais dados")
print("  - Ambos usam as mesmas features (média+std+min+max dos keypoints)")
print("  - O MLP precisa de normalização; o XGBoost não")

---

## 9. MLP para Classificação de Execução (binária)

O AlignFit também classifica se a execução foi correta ou errada. Essa é uma tarefa binária. Mostramos aqui como o MLP se comporta nesse cenário — e por que **Sigmoid** (não Softmax) é usado na saída binária.

In [ ]:
# =========================================================
# CLASSIFICAÇÃO BINÁRIA DE EXECUÇÃO
# Exemplo com um único exercício (bíceps)
# =========================================================

TARGET_EXERCISE = "biceps"   # troque para testar outro exercício

df_exec = df[df["tipo_exercicio"] == TARGET_EXERCISE].copy()

df_exec["label_exec"] = df_exec["tipo_execucao"].map({"correta": 0, "errada": 1})

X_exec = []
y_exec = []

for _, group in df_exec.groupby("video"):
    features = []
    features.extend(group[feature_columns].mean().tolist())
    features.extend(group[feature_columns].std().fillna(0).tolist())
    features.extend(group[feature_columns].min().tolist())
    features.extend(group[feature_columns].max().tolist())
    X_exec.append(features)
    y_exec.append(group["label_exec"].iloc[0])

X_exec = np.array(X_exec)
y_exec = np.array(y_exec)

print(f"Exercício: {TARGET_EXERCISE}")
print(f"Total de vídeos: {len(y_exec)}")
print(f"  Correta (0): {(y_exec==0).sum()}")
print(f"  Errada  (1): {(y_exec==1).sum()}")

X_tr_e, X_te_e, y_tr_e, y_te_e = train_test_split(
    X_exec, y_exec, test_size=0.2, random_state=42, stratify=y_exec
)

scaler_exec = StandardScaler()
X_tr_e_sc = scaler_exec.fit_transform(X_tr_e)
X_te_e_sc = scaler_exec.transform(X_te_e)

# =========================================================
# MLP BINÁRIO
# Para classificação binária, o sklearn ainda usa
# internamente Softmax de 2 classes (equivalente ao Sigmoid)
# =========================================================

mlp_exec = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=0.001,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=15,
    random_state=42
)

mlp_exec.fit(X_tr_e_sc, y_tr_e)

y_pred_exec = mlp_exec.predict(X_te_e_sc)

print(f"\nResultados — MLP Binário ({TARGET_EXERCISE}):")
print(classification_report(y_te_e, y_pred_exec, target_names=["correta", "errada"]))

---

## 10. Salvando o Modelo MLP

In [ ]:
import joblib

# =========================================================
# SALVAR MODELO E SCALER
# O scaler precisa ser salvo junto — na inferência,
# a normalização deve ser idêntica à do treino.
# =========================================================

MLP_MODEL_OUTPUT   = r"..\models\modelo-mlp-softmax-classificacao.pkl"
MLP_SCALER_OUTPUT  = r"..\models\scaler-mlp-softmax-classificacao.pkl"
MLP_ENCODER_OUTPUT = r"..\models\encoder-mlp-softmax-classificacao.pkl"

joblib.dump(mlp,           MLP_MODEL_OUTPUT)
joblib.dump(scaler,        MLP_SCALER_OUTPUT)
joblib.dump(label_encoder, MLP_ENCODER_OUTPUT)

print("Modelos salvos:")
print(f"  Modelo MLP:   {MLP_MODEL_OUTPUT}")
print(f"  Scaler:       {MLP_SCALER_OUTPUT}")
print(f"  LabelEncoder: {MLP_ENCODER_OUTPUT}")

# =========================================================
# VERIFICAÇÃO — carregar e rodar inferência de teste
# =========================================================

mlp_carregado           = joblib.load(MLP_MODEL_OUTPUT)
scaler_carregado        = joblib.load(MLP_SCALER_OUTPUT)
label_encoder_carregado = joblib.load(MLP_ENCODER_OUTPUT)

# Pegar uma amostra aleatória do teste para verificar
idx_teste = 0
amostra   = X_test[idx_teste].reshape(1, -1)
amostra_normalizada = scaler_carregado.transform(amostra)

proba_carregado = mlp_carregado.predict_proba(amostra_normalizada)[0]
pred_carregado  = mlp_carregado.predict(amostra_normalizada)[0]

print("\nVerificação do modelo carregado:")
print(f"  Classe real:    {label_encoder_carregado.classes_[y_test[idx_teste]]}")
print(f"  Classe prevista: {label_encoder_carregado.classes_[pred_carregado]}")
print(f"  Probabilidades Softmax:")
for cls, p in zip(label_encoder_carregado.classes_, proba_carregado):
    print(f"    {cls:20s}: {p:.4f} ({p*100:.1f}%)")

---

## 11. Resumo Conceitual

### O que foi implementado

| Componente | Papel no modelo |
|---|---|
| **Input Layer** | Recebe as 136 features (média+std+min+max dos 34 keypoints x/y) |
| **Hidden Layers (ReLU)** | Aprende representações não-lineares dos padrões de movimento |
| **Output Layer (Softmax)** | Converte os scores finais em probabilidades por exercício |
| **Cross-Entropy Loss** | Penaliza quando a probabilidade da classe correta é baixa |
| **Adam Optimizer** | Ajusta os pesos via backpropagation com taxa de aprendizado adaptativa |
| **StandardScaler** | Normaliza as features para a mesma escala (essencial para MLP) |
| **Early Stopping** | Para o treino quando a validação para de melhorar (evita overfitting) |

### Conexão com o Perceptron da professora

```
Perceptron (MY_FIRST_PERCEPTRON.ipynb)
  ↓ 1 neurônio, função de passo, atualização manual
  ↓ só classificação binária, não diferenciável
  
MLP com Softmax (este notebook)
  ↓ múltiplas camadas, ReLU + Softmax, backpropagation
  ↓ classificação multiclasse, diferenciável, treina por gradiente
```

### Por que Softmax para o AlignFit?

Temos **4 exercícios mutuamente exclusivos** — um vídeo é de exatamente um exercício. O Softmax garante que as 4 probabilidades somem 1 e que a classe com maior score seja claramente destacada. É matematicamente consistente com Cross-Entropy e torna a saída interpretável: "92% de chance de ser desenvolvimento de ombro".